# BiHT-Style Shared-Topic Stream Join Simulation

This notebook extends the shared-topic Spark Structured Streaming example by simulating a **Bit-vector Hash Table (BiHT)** idea inspired by AMJoin.

The goal is teaching-oriented:

- both producers write to the same Kafka topic,
- Spark reads the shared topic once,
- each record is identified by `stream_id`,
- a small in-memory BiHT-like table tracks which streams have appeared for each join key,
- the notebook prints joined pairs only when the bit-vector shows that both required streams are present.

This is a simple simulation to show the idea of using bit-vectors to avoid unnecessary join attempts.

## AMJoin / BiHT Intuition

In AMJoin, each hash-table entry has a bit-vector.

For two streams:

| Bit | Meaning |
|---:|---|
| bit 0 | Stream A has at least one tuple for this key |
| bit 1 | Stream B has at least one tuple for this key |

For this notebook:

```text
01 means only Stream A has appeared
10 means only Stream B has appeared
11 means both streams have appeared, so a join may be possible
```

The join key in this simple example is `number`.

In [1]:
import os
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructField, StructType, StringType, IntegerType

## Spark And Kafka Configuration

This version follows the same API style as the earlier PySpark examples.

It is written for PySpark `3.3.0` and Kafka `2.8.2`.

In [2]:
HOST_IP = "10.192.8.223"
KAFKA_BOOTSTRAP_SERVERS = f"{HOST_IP}:9092"
SHARED_TOPIC = "teaching-stream-shared"

SPARK_PACKAGES = (
    "org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SPARK_PACKAGES} pyspark-shell"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Teaching Example - BiHT-Style Shared Topic Join")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark session started.")

Spark session started.


## Define The Shared Topic Schema

Both producers should send JSON records with the same schema.

Example Producer A message:

```python
{
    "event_id": "...",
    "stream_id": "A",
    "number": 3,
    "event_time": "2026-05-12T10:00:00.123456"
}
```

Example Producer B message:

```python
{
    "event_id": "...",
    "stream_id": "B",
    "number": 3,
    "event_time": "2026-05-12T10:00:05.654321"
}
```

In [3]:
event_schema = StructType([
    StructField("event_id", StringType(), nullable=False),
    StructField("stream_id", StringType(), nullable=False),
    StructField("number", IntegerType(), nullable=False),
    StructField("event_time", StringType(), nullable=False)
])

## Read And Parse The Shared Kafka Topic

Spark reads one topic containing both producers' records.

The parsed stream contains:

- `event_id`
- `stream_id`
- `number`
- `event_time`
- `event_timestamp`

In [4]:
shared_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", SHARED_TOPIC)
    .option("startingOffsets", "latest")
    .option("failOnDataLoss", "false")
    .load()
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(from_json(col("json_value"), event_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp", to_timestamp(col("event_time")))
    .filter(col("stream_id").isin("A", "B"))
)

print("Shared Kafka stream created.")

Shared Kafka stream created.


## BiHT Simulation State

The following Python dictionaries are maintained on the Jupyter driver for teaching purposes.

For each join key, `biht[number]` stores a bit mask:

```text
1 = Stream A present
2 = Stream B present
3 = both Stream A and Stream B present
```

The `event_buffers` dictionary keeps recent events for each key so that matched records can be printed.

Again: this driver-side state is for teaching. In a production distributed Spark job, state should be managed using Spark's stateful streaming mechanisms or an external store.

In [5]:
STREAM_BITS = {
    "A": 0b01,
    "B": 0b10
}

FULL_MATCH_MASK = 0b11
JOIN_WINDOW_SECONDS = 10
STATE_TTL_SECONDS = 60

# BiHT-like structure:
# key: number
# value: bit mask showing which streams have appeared for this number
biht = {}

# Recent events grouped by join key and source stream.
# Example:
# event_buffers[3]["A"] = [event_from_A_1, event_from_A_2]
# event_buffers[3]["B"] = [event_from_B_1]
event_buffers = {}

## Helper Functions

These functions:

- format bit-vectors for display,
- remove old buffered events,
- check whether two events are close enough in time,
- print joined pairs.

In [6]:
def bitmask_to_string(mask):
    return format(mask, "02b")


def event_to_dict(row):
    return {
        "event_id": row.event_id,
        "stream_id": row.stream_id,
        "number": row.number,
        "event_timestamp": row.event_timestamp
    }


def is_within_join_window(event_left, event_right):
    delta = abs((event_left["event_timestamp"] - event_right["event_timestamp"]).total_seconds())
    return delta <= JOIN_WINDOW_SECONDS


def prune_old_state(current_time):
    expiry_time = current_time - timedelta(seconds=STATE_TTL_SECONDS)
    keys_to_delete = []

    for number, by_stream in event_buffers.items():
        for stream_id in list(by_stream.keys()):
            by_stream[stream_id] = [
                event for event in by_stream[stream_id]
                if event["event_timestamp"] is not None and event["event_timestamp"] >= expiry_time
            ]

        # Recompute the bit-vector after pruning.
        new_mask = 0
        for stream_id, events in by_stream.items():
            if events:
                new_mask |= STREAM_BITS[stream_id]

        if new_mask == 0:
            keys_to_delete.append(number)
        else:
            biht[number] = new_mask

    for number in keys_to_delete:
        event_buffers.pop(number, None)
        biht.pop(number, None)


def print_joined_pairs(number, new_event):
    opposite_stream = "B" if new_event["stream_id"] == "A" else "A"
    opposite_events = event_buffers.get(number, {}).get(opposite_stream, [])

    matches = [
        old_event for old_event in opposite_events
        if is_within_join_window(new_event, old_event)
    ]

    if not matches:
        print(
            f"  BiHT says both streams exist for key={number}, "
            "but no opposite event is inside the time window."
        )
        return

    for old_event in matches:
        event_a = new_event if new_event["stream_id"] == "A" else old_event
        event_b = new_event if new_event["stream_id"] == "B" else old_event

        print("  JOINED RESULT")
        print(f"    number        : {number}")
        print(f"    event_id_A    : {event_a['event_id']}")
        print(f"    event_time_A  : {event_a['event_timestamp']}")
        print(f"    event_id_B    : {event_b['event_id']}")
        print(f"    event_time_B  : {event_b['event_timestamp']}")

## Process Each Micro-Batch

This is where the BiHT-style logic is simulated.

For each incoming event:

1. Use `number` as the join key.
2. Set the relevant bit for Stream A or Stream B.
3. Check the bit-vector.
4. If the bit-vector is not `11`, do not attempt a join.
5. If the bit-vector is `11`, check buffered events and print joined pairs within the time window.

This mirrors the AMJoin idea: use the bit-vector as a fast test before doing more expensive join work.

In [7]:
def process_batch_with_biht(batch_df, batch_id):
    rows = batch_df.orderBy("event_timestamp").collect()

    if not rows:
        print(f"Batch {batch_id}: no new events")
        return

    print(f"\n========== Batch {batch_id}: {len(rows)} event(s) ==========")

    latest_time = max(row.event_timestamp for row in rows if row.event_timestamp is not None)
    if latest_time is not None:
        prune_old_state(latest_time)

    for row in rows:
        event = event_to_dict(row)
        number = event["number"]
        stream_id = event["stream_id"]

        if event["event_timestamp"] is None:
            print(f"Skipping event with invalid timestamp: {event}")
            continue

        event_buffers.setdefault(number, {"A": [], "B": []})
        event_buffers[number][stream_id].append(event)

        previous_mask = biht.get(number, 0)
        new_mask = previous_mask | STREAM_BITS[stream_id]
        biht[number] = new_mask

        print(
            f"Event from Stream {stream_id}: number={number}, "
            f"BiHT {bitmask_to_string(previous_mask)} -> {bitmask_to_string(new_mask)}"
        )

        if new_mask != FULL_MATCH_MASK:
            print("  Join skipped: not all required stream bits are present yet.")
        else:
            print("  Join possible: both Stream A and Stream B bits are present.")
            print_joined_pairs(number, event)

    print("\nCurrent BiHT state:")
    for number in sorted(biht.keys()):
        print(f"  key={number}: bit-vector={bitmask_to_string(biht[number])}")

## Start The BiHT-Style Streaming Query

This query uses `foreachBatch` so the notebook can print the simulated BiHT state and joined records.

Leave this cell running while Producer A and Producer B are running.

In [ ]:
query = (
    shared_stream.writeStream
    .foreachBatch(process_batch_with_biht)
    .outputMode("append")
    .option("checkpointLocation", "checkpoint_biht_shared_topic_example")
    .start()
)

print("BiHT-style shared-topic streaming query started.")
query.awaitTermination()

BiHT-style shared-topic streaming query started.
Batch 0: no new events

========== Batch 1: 2 event(s) ==========
Event from Stream B: number=5, BiHT 00 -> 10
  Join skipped: not all required stream bits are present yet.
Event from Stream A: number=3, BiHT 00 -> 01
  Join skipped: not all required stream bits are present yet.

Current BiHT state:
  key=3: bit-vector=01
  key=5: bit-vector=10

========== Batch 2: 2 event(s) ==========
Event from Stream B: number=2, BiHT 00 -> 10
  Join skipped: not all required stream bits are present yet.
Event from Stream A: number=5, BiHT 10 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 5
    event_id_A    : ba9f2c65-dd5f-4b3a-91f9-0abac30b1c9e
    event_time_A  : 2026-05-12 05:29:10.291094
    event_id_B    : 02f470ca-37ce-4707-9650-0f0f6caf22e3
    event_time_B  : 2026-05-12 05:29:08.283214

Current BiHT state:
  key=2: bit-vector=10
  key=3: bit-vector=01
  key=5: bit-vector=11

==========


========== Batch 12: 2 event(s) ==========
Event from Stream B: number=1, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  BiHT says both streams exist for key=1, but no opposite event is inside the time window.
Event from Stream A: number=1, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 1
    event_id_A    : 8f8a7004-a168-4420-9fbb-992c56ebae8a
    event_time_A  : 2026-05-12 05:29:30.190968
    event_id_B    : fc6b718d-5baa-42fd-acf4-e109739d2754
    event_time_B  : 2026-05-12 05:29:30.189586

Current BiHT state:
  key=1: bit-vector=11
  key=2: bit-vector=11
  key=3: bit-vector=11
  key=4: bit-vector=11
  key=5: bit-vector=11

========== Batch 13: 2 event(s) ==========
Event from Stream B: number=3, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 3
    event_id_A    : 85aeae76-8ca6-4d11-b74d-b4d3b6491f1e
    event_time_A  : 202


========== Batch 21: 2 event(s) ==========
Event from Stream A: number=2, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 2
    event_id_A    : 3ea6fde5-3820-41d6-9770-bb253b61e885
    event_time_A  : 2026-05-12 05:29:48.241181
    event_id_B    : bc5fa66e-ccdc-4ac3-8cac-73524902e952
    event_time_B  : 2026-05-12 05:29:42.224894
  JOINED RESULT
    number        : 2
    event_id_A    : 3ea6fde5-3820-41d6-9770-bb253b61e885
    event_time_A  : 2026-05-12 05:29:48.241181
    event_id_B    : c1900068-39fb-4889-969c-53719a19fdbe
    event_time_B  : 2026-05-12 05:29:46.237378
Event from Stream B: number=4, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 4
    event_id_A    : da233a5c-4fdd-4e0b-8876-19a5df8dae0e
    event_time_A  : 2026-05-12 05:29:46.235224
    event_id_B    : 505dafe2-df4c-43cd-ab2b-6b3c949514f7
    event_time_B  : 2026-05-12 05:29:48.242054

Cur


========== Batch 31: 1 event(s) ==========
Event from Stream B: number=3, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 3
    event_id_A    : 46b49e1e-360a-43dc-ba22-2bb1f35446f6
    event_time_A  : 2026-05-12 05:29:56.263360
    event_id_B    : f05c6764-f784-47b3-b47a-986d61374cc3
    event_time_B  : 2026-05-12 05:30:06.134745
  JOINED RESULT
    number        : 3
    event_id_A    : 40f77f06-4502-4d15-9c4b-206d39941051
    event_time_A  : 2026-05-12 05:30:00.119838
    event_id_B    : f05c6764-f784-47b3-b47a-986d61374cc3
    event_time_B  : 2026-05-12 05:30:06.134745
  JOINED RESULT
    number        : 3
    event_id_A    : a9176b4b-ae14-493d-bce5-d6f7c688e5eb
    event_time_A  : 2026-05-12 05:30:02.123242
    event_id_B    : f05c6764-f784-47b3-b47a-986d61374cc3
    event_time_B  : 2026-05-12 05:30:06.134745
  JOINED RESULT
    number        : 3
    event_id_A    : f9a8b8cc-629a-48e9-a458-cb9cdf4b7c8c
    event_time_A


========== Batch 41: 2 event(s) ==========
Event from Stream B: number=1, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  BiHT says both streams exist for key=1, but no opposite event is inside the time window.
Event from Stream A: number=4, BiHT 11 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
    number        : 4
    event_id_A    : d76e7b64-bdd1-4d65-a585-0e5a8ecf273c
    event_time_A  : 2026-05-12 05:30:24.186033
    event_id_B    : f5b512a3-ba41-47c3-8cb2-8cff3a9cf89a
    event_time_B  : 2026-05-12 05:30:16.163160
  JOINED RESULT
    number        : 4
    event_id_A    : d76e7b64-bdd1-4d65-a585-0e5a8ecf273c
    event_time_A  : 2026-05-12 05:30:24.186033
    event_id_B    : 13a6b40d-449a-4b93-97db-0f32cbf152c3
    event_time_B  : 2026-05-12 05:30:20.174426

Current BiHT state:
  key=1: bit-vector=11
  key=2: bit-vector=11
  key=3: bit-vector=11
  key=4: bit-vector=11
  key=5: bit-vector=11

========== Batch 42: 2

## Stop The Query

If you interrupt the previous cell, run this cell to stop the streaming query.

In [ ]:
try:
    query.stop()
    print("Streaming query stopped.")
except NameError:
    print("No active query object found.")

## How To Interpret The Output

Example:

```text
Event from Stream A: number=3, BiHT 00 -> 01
  Join skipped: not all required stream bits are present yet.

Event from Stream B: number=3, BiHT 01 -> 11
  Join possible: both Stream A and Stream B bits are present.
  JOINED RESULT
```

This means:

- the first event created a partial state for key `3`,
- the second event completed the bit-vector for key `3`,
- only then did the notebook attempt to find matching events within the time window.

This demonstrates the core AMJoin idea in a simplified way.